# BugScanner YOLO11 Vietnam Practical Pest Training

This notebook is designed for Kaggle with Internet disabled. Add these inputs before running:

- `trungminh815/bugscanner-vietnam-practical-yolo`
- `trungminh815/bugscanner-yolo-training-code`
- `trungminh815/bugscanner-yolo11-weights`
- `trungminh815/bugscanner-offline-wheels`

This trains the practical 33-class Vietnam agriculture pest dataset, not the original 102-class IP102 species dataset.


## 1. Config
Edit this cell to control the Kaggle inputs and YOLO training parameters.


In [ ]:
from pathlib import Path
import subprocess
import sys

# ============================================================
# 1. CONFIG - edit this cell before running the notebook.
# ============================================================

# Run mode
SMOKE_TEST = False

# Kaggle input dataset slugs
DATASET_SLUG = 'bugscanner-vietnam-practical-yolo'
CODE_SLUG = 'bugscanner-yolo-training-code'
WEIGHTS_SLUG = 'bugscanner-yolo11-weights'
WHEELS_SLUG = 'bugscanner-offline-wheels'
MODEL_WEIGHTS_FILE = 'yolo11m.pt'

# Training run identity
MODEL_NAME = 'yolo11m'
IMG_SIZE = 896
RUN_NAME = f'{MODEL_NAME}-vn-practical-{IMG_SIZE}-v3-dataqa' + ('-smoke' if SMOKE_TEST else '')

# Core training controls
EPOCHS = 1 if SMOKE_TEST else 120
BATCH = 8
DEVICE = 0
WORKERS = 2
SEED = 42
PATIENCE = 30

# Optimizer and schedule controls
OPTIMIZER = 'auto'
LR0 = 0.01
LRF = 0.01
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
COS_LR = True

# Training behavior controls
AMP = True
CACHE = False
CLOSE_MOSAIC = 10
SAVE_PERIOD = -1
PLOTS = True
EXIST_OK = True

# Add any extra Ultralytics CLI args here, for example:
# EXTRA_TRAIN_ARGS = ['degrees=5.0', 'translate=0.1', 'scale=0.5']
EXTRA_TRAIN_ARGS = []

# Notebook logging controls
# 'summary' writes full YOLO output to TRAIN_LOG_FILE and only prints compact progress.
# 'raw' prints the normal YOLO progress bar directly in the notebook.
LOG_MODE = 'summary'
LOG_POLL_SECONDS = 60
LOG_TAIL_LINES = 80

# Paths. Usually no need to edit these.
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = INPUT_ROOT / 'datasets' / 'trungminh815'
WORK_ROOT = Path('/kaggle/working')
PROJECT_DIR = WORK_ROOT / 'runs' / 'train'
WORK_DATA_YAML = WORK_ROOT / 'vietnam-practical-kaggle.yaml'


def input_path(slug):
    candidates = [INPUT_ROOT / slug, DATASET_ROOT / slug]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


DATASET_INPUT = input_path(DATASET_SLUG)
CODE_INPUT = input_path(CODE_SLUG)
WEIGHTS_INPUT = input_path(WEIGHTS_SLUG) / MODEL_WEIGHTS_FILE
WHEELS_INPUT = input_path(WHEELS_SLUG)

print('SMOKE_TEST =', SMOKE_TEST)
print('RUN_NAME =', RUN_NAME)
print('EPOCHS =', EPOCHS, 'BATCH =', BATCH, 'IMG_SIZE =', IMG_SIZE, 'DEVICE =', DEVICE)
print('DATASET_INPUT =', DATASET_INPUT)
print('CODE_INPUT =', CODE_INPUT)
print('WEIGHTS_INPUT =', WEIGHTS_INPUT)
print('WHEELS_INPUT =', WHEELS_INPUT)


## 2. Input Check
Verify that the Kaggle datasets, offline wheels, and starting weights are mounted correctly.


In [ ]:
required_paths = [DATASET_INPUT, CODE_INPUT, WEIGHTS_INPUT, WHEELS_INPUT]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Kaggle input(s): ' + ', '.join(missing))

wheel_files = sorted(WHEELS_INPUT.glob('*.whl'))
print('Offline wheels:', len(wheel_files))
for wheel in wheel_files[:20]:
    print(' -', wheel.name)
if not wheel_files:
    raise FileNotFoundError('No offline wheels found in ' + str(WHEELS_INPUT))


## 3. Offline Package Install
Install Ultralytics from the uploaded wheel dataset with internet disabled.


In [ ]:
def run(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--no-index',
    '--find-links',
    WHEELS_INPUT,
    '--no-deps',
    'ultralytics==8.4.86',
    'ultralytics-thop',
    'nvidia-ml-py',
    'polars',
    'polars-runtime-32',
])


## 4. Model And Runtime Check
Confirm PyTorch, CUDA, GPU name, Ultralytics version, and the starting YOLO weight file.


In [ ]:
import torch
import ultralytics
import yaml

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('ultralytics:', ultralytics.__version__)
assert ultralytics.__version__ == '8.4.86'
assert WEIGHTS_INPUT.exists(), WEIGHTS_INPUT


## 5. Dataset Prepare
Create the Kaggle-specific 33-class practical pest YOLO data YAML and verify image/label counts for train, val, and test splits.


In [ ]:
source_yaml = CODE_INPUT / 'configs' / 'vietnam-practical-yolo.yaml'
with source_yaml.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = str(DATASET_INPUT)
data_cfg['train'] = 'images/train'
data_cfg['val'] = 'images/val'
data_cfg['test'] = 'images/test'
with WORK_DATA_YAML.open('w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

for split in ['train', 'val', 'test']:
    images = sorted((DATASET_INPUT / 'images' / split).glob('*'))
    labels = sorted((DATASET_INPUT / 'labels' / split).glob('*.txt'))
    print(split, 'images=', len(images), 'labels=', len(labels))
    if not images or not labels:
        raise RuntimeError(f'Missing images or labels for split {split}')

print(WORK_DATA_YAML.read_text(encoding='utf-8')[:1000])


## 6. Dataset QA
Print metadata reports uploaded with the Kaggle dataset so weak-class coverage is visible before training.


In [ ]:
import csv

metadata_dir = DATASET_INPUT / 'metadata'
class_map_csv = metadata_dir / 'class_map.csv'
balance_report = metadata_dir / 'balance_report.md'
image_manifest = metadata_dir / 'image_manifest.csv'

if class_map_csv.exists():
    with class_map_csv.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    weak_rows = []
    for row in rows:
        train_images = int(row.get('train_images') or 0)
        val_images = int(row.get('val_images') or 0)
        test_images = int(row.get('test_images') or 0)
        if train_images < 300 or val_images < 30 or test_images < 30:
            weak_rows.append((train_images, val_images, test_images, row['id'], row['key'], row['name']))
    print('Dataset classes:', len(rows))
    print('Classes below v3 QA targets:', len(weak_rows))
    for train_images, val_images, test_images, class_id, key, name in sorted(weak_rows)[:20]:
        print(f'{class_id:>2} {key:<28} train={train_images:<4} val={val_images:<4} test={test_images:<4} {name}')
else:
    print('No class_map.csv found at', class_map_csv)

if image_manifest.exists():
    with image_manifest.open('r', encoding='utf-8') as f:
        image_rows = sum(1 for _ in f) - 1
    print('Image manifest rows:', image_rows)
else:
    print('No image_manifest.csv found at', image_manifest)

if balance_report.exists():
    print(balance_report.read_text(encoding='utf-8')[:2000])


## 7. Training
Run YOLO detection training using the parameters from the config section.


In [ ]:
import csv
import time

train_args = {
    'model': WEIGHTS_INPUT,
    'data': WORK_DATA_YAML,
    'epochs': EPOCHS,
    'batch': BATCH,
    'imgsz': IMG_SIZE,
    'device': DEVICE,
    'workers': WORKERS,
    'seed': SEED,
    'patience': PATIENCE,
    'optimizer': OPTIMIZER,
    'lr0': LR0,
    'lrf': LRF,
    'momentum': MOMENTUM,
    'weight_decay': WEIGHT_DECAY,
    'warmup_epochs': WARMUP_EPOCHS,
    'cos_lr': COS_LR,
    'amp': AMP,
    'cache': CACHE,
    'close_mosaic': CLOSE_MOSAIC,
    'save_period': SAVE_PERIOD,
    'project': PROJECT_DIR,
    'name': RUN_NAME,
    'exist_ok': EXIST_OK,
    'plots': PLOTS,
}

train_cmd = ['yolo', 'detect', 'train']
for key, value in train_args.items():
    if value is not None:
        train_cmd.append(f'{key}={value}')
train_cmd.extend(EXTRA_TRAIN_ARGS)

run_dir = PROJECT_DIR / RUN_NAME
results_csv = run_dir / 'results.csv'
TRAIN_LOG_FILE = WORK_ROOT / f'{RUN_NAME}-train.log'


def read_latest_results():
    if not results_csv.exists():
        return None
    with results_csv.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return None
    row = {k.strip(): v for k, v in rows[-1].items()}
    keys = [
        'epoch',
        'train/box_loss',
        'train/cls_loss',
        'train/dfl_loss',
        'metrics/precision(B)',
        'metrics/recall(B)',
        'metrics/mAP50(B)',
        'metrics/mAP50-95(B)',
        'val/box_loss',
        'val/cls_loss',
        'val/dfl_loss',
        'lr/pg0',
    ]
    parts = []
    for key in keys:
        value = row.get(key)
        if value not in (None, ''):
            parts.append(f'{key}={value}')
    return ' | '.join(parts) if parts else str(row)


def print_log_tail(path, lines=80):
    if not path.exists():
        print('No training log found at', path)
        return
    text = path.read_text(encoding='utf-8', errors='replace')
    clean = text.replace('\r', '\n')
    tail = clean.splitlines()[-lines:]
    print(f'Last {len(tail)} log lines from {path}:')
    print('\n'.join(tail))


if LOG_MODE == 'raw':
    run(train_cmd)
elif LOG_MODE == 'summary':
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    print('+', ' '.join(map(str, train_cmd)))
    print('Full raw YOLO log:', TRAIN_LOG_FILE)
    with TRAIN_LOG_FILE.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            list(map(str, train_cmd)),
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
        last_summary = None
        while process.poll() is None:
            time.sleep(LOG_POLL_SECONDS)
            summary = read_latest_results()
            if summary and summary != last_summary:
                print(time.strftime('%Y-%m-%d %H:%M:%S'), summary)
                last_summary = summary
            else:
                print(time.strftime('%Y-%m-%d %H:%M:%S'), 'training still running...')

    if process.returncode != 0:
        print_log_tail(TRAIN_LOG_FILE, LOG_TAIL_LINES)
        raise subprocess.CalledProcessError(process.returncode, train_cmd)

    summary = read_latest_results()
    if summary:
        print('Final:', summary)
    print('Training completed. Full raw YOLO log:', TRAIN_LOG_FILE)
else:
    raise ValueError(f'Unknown LOG_MODE: {LOG_MODE}')


## 8. Output Check
Verify that the trained weights and training metrics were written to the run directory.


In [ ]:
run_dir = PROJECT_DIR / RUN_NAME
for rel in ['weights/best.pt', 'weights/last.pt', 'results.csv']:
    path = run_dir / rel
    print(rel, path.exists(), path.stat().st_size if path.exists() else None)
    if not path.exists():
        raise FileNotFoundError(path)

print('Run directory:', run_dir)
print('Done')


## 9. Fixed Test Evaluation
Evaluate the best checkpoint on the unchanged test split for fair v2/v3 comparison.


In [ ]:
TEST_LOG_FILE = WORK_ROOT / f'{RUN_NAME}-test.log'
best_weights = PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt'
test_cmd = [
    'yolo', 'detect', 'val',
    f'model={best_weights}',
    f'data={WORK_DATA_YAML}',
    'split=test',
    f'imgsz={IMG_SIZE}',
    f'batch={BATCH}',
    f'device={DEVICE}',
]
print('+', ' '.join(map(str, test_cmd)))
with TEST_LOG_FILE.open('w', encoding='utf-8') as log:
    subprocess.run(test_cmd, check=True, stdout=log, stderr=subprocess.STDOUT, text=True)
print_log_tail(TEST_LOG_FILE, LOG_TAIL_LINES)
print('Test evaluation log:', TEST_LOG_FILE)
